In [1]:
from decouple import AutoConfig
config = AutoConfig(search_path='./../.env')

In [2]:
import pprint

In [3]:
import os
import openai

openai.api_key = config('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = openai.api_key

### Defining the LLM

In [ ]:
from langchain_openai import ChatOpenAI
model_name = 'gpt-4o-mini'
llm = ChatOpenAI(
    model=model_name,
    temperature=0.0,
    max_tokens=1024
)

llm

### Defining the Graph state

State is the object that is passed between nodes in the graph.

In [5]:
from typing import TypedDict, Annotated, List
from langchain_core.messages import BaseMessage, AnyMessage
import operator
from IPython.display import Image, display

In [6]:
class AgentState(TypedDict):
    input: str
    agent_outcome: List[AnyMessage]
    chat_history: Annotated[list, operator.add]

### Define the agent (node)

In [7]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

def research_agent(data):
    print(data)
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "You are a helpful AI assistant"
                "\nUser Query: {input}"
            ),
            MessagesPlaceholder(variable_name="chat_history"),
        ]
    )
    agent = prompt | llm
    result = agent.invoke(data)
    return {
                'agent_outcome': [result],
                'chat_history': ["User: " + data['input'] + "\n AI Message: " + result.content],
            }

### Defining the workflow (graph)

In [ ]:
from langgraph.graph import END, StateGraph

## Initialising the workflow
workflow = StateGraph(AgentState)

## Adding node (agent) to the graph (workflow)
workflow.add_node("research", research_agent)

## Setting the entry point of the graph
workflow.set_entry_point("research")

## Compiling the graph
app = workflow.compile()
try:
    display(Image(app.get_graph(xray=True).draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
inputs = {
    "input": "What are Small Language Models?",
}

app.invoke(input=inputs)

`invoke()` method is used to run the workflow. It runs the entire graph (workflow) synchronously. 

It takes the input data and returns the final state of the workflow. 

The final state contains the output of the last agent that was executed in the workflow.

In [ ]:
for s in app.stream(input=inputs):
    pprint.pp(s)
    print(list(s.values())[0]['agent_outcome'][0].content)
    print("-----"*20)

In [ ]:
def stream_app_updates(user_input: str, chat_history: list):
    inputs = {
        "input": user_input,
        "chat_history": chat_history
    }
    for event in app.stream(input=inputs):
        for value in event.values():
            response = value["agent_outcome"][-1].content
            conv = value["chat_history"][-1]
    return response, conv

chat_history = []
while True:
    try:
        user_input = input("User: ")
        if user_input.lower() in ["quit", "exit", "q"]:
            print("Goodbye!")
            break

        response, conv = stream_app_updates(user_input, chat_history)
    except:
        # fallback if input() is not available
        print("User: " + user_input)
        response, conv = stream_app_updates(user_input, chat_history)
        break
    print("Assistant:", response)
    chat_history.append(conv)